# Coffee Detection with Standard — Roboflow to shared Drive

Mengunduh `tes-rcphs/coffee-detection-with-standard` versi 8 langsung di Colab, mengaudit struktur dasarnya, lalu menyimpan satu arsip persisten ke folder proyek bersama. Notebook ini tidak melakukan training dan tidak membuka dataset Faruq/Adrian/test penelitian utama.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, shutil, subprocess, sys
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/af2-igem-paired-confirmation'
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'roboflow'], check=True)
os.chdir(REPO)
import importlib
sys.path.insert(0, str(REPO/'src'))
importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
from coffee_detector.drive_project import resolve_drive_project_root
PROJECT_ROOT = resolve_drive_project_root()
print('PROJECT_ROOT:', PROJECT_ROOT)


In [ ]:
from google.colab import userdata
from roboflow import Roboflow

WORKSPACE = 'tes-rcphs'
PROJECT = 'coffee-detection-with-standard'
VERSION = 8
FORMAT = 'yolov8'
LOCAL_ROOT = Path('/content/coffee-detection-with-standard-v8')
BUNDLE = PROJECT_ROOT / 'bundles/coffee-detection-with-standard-v8-yolov8.tar'
EVIDENCE = PROJECT_ROOT / 'evidence/coffee-detection-with-standard-v8'
EVIDENCE.mkdir(parents=True, exist_ok=True)
BUNDLE.parent.mkdir(parents=True, exist_ok=True)

if BUNDLE.is_file():
    print('SUDAH ADA, download dilewati:', BUNDLE)
else:
    api_key = userdata.get('ROBOFLOW_API_KEY')
    assert api_key, 'Tambahkan Colab Secret ROBOFLOW_API_KEY dan aktifkan akses notebook.'
    if LOCAL_ROOT.exists(): shutil.rmtree(LOCAL_ROOT)
    rf = Roboflow(api_key=api_key)
    version = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION)
    dataset = version.download(FORMAT, location=str(LOCAL_ROOT), overwrite=True)
    print('DOWNLOAD:', dataset.location)


In [ ]:
import hashlib, json, tarfile, time, yaml

if not BUNDLE.is_file():
    yaml_files = sorted(LOCAL_ROOT.rglob('data.yaml'))
    assert len(yaml_files) == 1, f'data.yaml harus tepat satu, ditemukan: {yaml_files}'
    data_yaml = yaml.safe_load(yaml_files[0].read_text(encoding='utf-8'))
    names = data_yaml.get('names', [])
    if isinstance(names, dict): names = [names[key] for key in sorted(names, key=lambda x: int(x))]
    split_rows = {}
    for split in ('train','valid','test'):
        images = list((LOCAL_ROOT/split/'images').glob('*')) if (LOCAL_ROOT/split/'images').is_dir() else []
        labels = list((LOCAL_ROOT/split/'labels').glob('*.txt')) if (LOCAL_ROOT/split/'labels').is_dir() else []
        instances = 0
        for label in labels:
            instances += sum(1 for row in label.read_text(errors='replace').splitlines() if row.strip())
        split_rows[split] = {'images': len(images), 'labels': len(labels), 'instances': instances}
    manifest = {
        'status': 'downloaded_unverified_external_dataset',
        'workspace': WORKSPACE, 'project': PROJECT, 'version': VERSION, 'format': FORMAT,
        'license_reported_by_public_page': 'CC BY 4.0',
        'downloaded_at_unix': int(time.time()), 'classes': names, 'class_count': len(names),
        'splits': split_rows,
        'research_warning': 'Do not train or compare before identity-duplicate, augmentation-sibling, ontology, and split audit.'
    }
    (EVIDENCE/'download_manifest.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
    with tarfile.open(BUNDLE, 'w') as archive:
        archive.add(LOCAL_ROOT, arcname=LOCAL_ROOT.name)
    digest = hashlib.sha256()
    with BUNDLE.open('rb') as handle:
        for chunk in iter(lambda: handle.read(8*1024*1024), b''): digest.update(chunk)
    manifest['archive'] = str(BUNDLE.relative_to(PROJECT_ROOT))
    manifest['archive_bytes'] = BUNDLE.stat().st_size
    manifest['archive_sha256'] = digest.hexdigest()
    (EVIDENCE/'download_manifest.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
else:
    manifest_path = EVIDENCE/'download_manifest.json'
    manifest = json.loads(manifest_path.read_text(encoding='utf-8')) if manifest_path.is_file() else {'archive': str(BUNDLE)}

print(json.dumps(manifest, indent=2, ensure_ascii=False))
print('BUNDLE  :', BUNDLE)
print('EVIDENCE:', EVIDENCE/'download_manifest.json')
print('SELESAI. Jangan training dahulu; kirim manifest untuk audit ontology dan leakage.')
